In [2]:
!pip install -q opencv-python mediapipe scikit-learn matplotlib

In [3]:
!pip install -q tensorflow

## Import Dependencies

In [4]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp

## preprocess
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

## model
import tensorflow as tf
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import LSTM, Dense, Input, Bidirectional, Dropout, LayerNormalization, TimeDistributed, Attention, Concatenate


## train eval
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import CategoricalAccuracy, Precision, Recall, TopKCategoricalAccuracy

In [6]:
mp_drawing = mp.solutions.drawing_utils
mp_holistic = mp.solutions.holistic
# mp_face_mesh = mp.solutions.face_mesh

## Detect and Draw Facial—Hands Landmarks

In [8]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) 
    image.flags.writeable = False                  
    results = model.process(image)                 
    image.flags.writeable = True                    
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) 
    return image, results

In [9]:
STYLES = {
    "left_hand": ((121, 22, 76), (121, 44, 250), 2, 4),
    "right_hand": ((245, 117, 66), (245, 66, 230), 2, 4)
}

def draw_styled_landmarks(image: any, results: any) -> None:
    landmarks_mapping = {
        "left_hand": (results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS),
        "right_hand": (results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS),
    }

    for key, (landmarks, connections) in landmarks_mapping.items():
        if landmarks:
            primary_color, secondary_color, thickness, radius = STYLES[key]
            mp_drawing.draw_landmarks(
                image, 
                landmarks, 
                connections,
                mp_drawing.DrawingSpec(color=primary_color, thickness=thickness, circle_radius=radius),
                mp_drawing.DrawingSpec(color=secondary_color, thickness=thickness, circle_radius=radius // 2)
            )

#### Test Video

In [9]:
cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        ret, frame = cap.read()

        frame = cv2.flip(frame, 1)

        image, results = mediapipe_detection(frame, holistic)
        
        draw_styled_landmarks(image, results)

        cv2.imshow('OpenCV Feed', image)

        if cv2.waitKey(10) & 0xFF == ord('q') or cv2.getWindowProperty('OpenCV Feed', cv2.WND_PROP_VISIBLE) < 1:
            break
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

## Set Up Image Datasets Directory

In [10]:
DATA_PATH = os.path.join('../dataset_final/dataset_final') 

label = open('../labels/label.txt', 'r').readline().split()

num_gestures = 120

sequence_length = 19

start_folder = 30

In [20]:
label

['hai',
 'nama',
 'kamu',
 'pagi',
 'siang',
 'malam',
 'siapa',
 'sudah',
 'belum',
 'makan',
 'suka',
 'selamat',
 'aku']

### Directories to Store datasets

In [91]:
for item in label:
    try:
        os.makedirs(os.path.join(DATA_PATH, item))
        for num_folder in range(0, num_gestures):
            try:
                os.makedirs(os.path.join(DATA_PATH, item, item+str(num_folder+1)))
            except:
                pass
    except:
        pass        

## Extract Datasets

so basically the below codes used to extract both hands landmarks using mediapipe as a detector, it will take 19 sequences of images and extract both hands landmarks. If it did not detect any hand landmarks then it will replace it with the existing ones taken in next iteration. This process will repeat 30 times. To get better result on the dataset, make sure that mediapipe detect your hands landmarks right when you are doing the gestures

In [93]:
label_extract = ['aku']

In [18]:
def extract_landmarks(results):
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([lh, rh])

In [95]:
cap = cv2.VideoCapture(0)

begin_extract = False

flag = 1

start_id = 60

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    for item in label_extract:
        for num_folder in range(start_id, start_id + start_folder):
            frame_num = 0
            while frame_num < sequence_length:

                ret, frame = cap.read()

                frame = cv2.flip(frame, 1)

                image, results = mediapipe_detection(frame, holistic)

                keypoints = extract_landmarks(results)

                draw_styled_landmarks(image, results)

                while not begin_extract:
                    ret, frame = cap.read()

                    frame = cv2.flip(frame, 1)

                    image, results = mediapipe_detection(frame, holistic)
                    
                    draw_styled_landmarks(image, results)
                    
                    cv2.rectangle(image, (50, 50), (380, 100), (0, 255, 0), -1)
                    cv2.putText(image, "Press 's' to Start Collecting", (60, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                    cv2.putText(image, f"collecting {item}", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                
                    cv2.imshow('Extract Dataset', image)

                    key_press = cv2.waitKey(1) & 0xFF
                    
                    if key_press == ord('q') or cv2.getWindowProperty('Extract Dataset', cv2.WND_PROP_VISIBLE) < 1:
                        flag=0
                        break

                    elif key_press == ord('s'):
                        begin_extract = True

                if not flag:
                    break

                cv2.putText(image, 'Collecting Sequences for {} Number {}'.format(item, num_folder+1), (15,12), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                cv2.imshow('Extract Dataset', image)

                ## if it did not detect any hand landmarks then skip 
                if (keypoints == 0).all() and begin_extract :
                    print(f"null vals on frame {frame_num+1} num folder {num_folder}")
                    continue
                
                npy_path = os.path.join(DATA_PATH, item, item+str(num_folder+1), str(frame_num+1))

                if os.path.exists(npy_path):
                    print(npy_path)
                    print("specified file exists")
                else:
                    np.save(npy_path, keypoints)

                ### increament iterator
                frame_num += 1
                
                key_press = cv2.waitKey(1) & 0xFF
                if key_press == ord('q') or key_press == 27 or cv2.getWindowProperty('Extract Dataset', cv2.WND_PROP_VISIBLE) < 1:   
                    flag=0
                    break
                    
            begin_extract = False
            if not flag:
                break
        if not flag:
            break
        
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(10)
    

null vals on frame 1 num folder 60


## Build Model

In [20]:
def f1_score(y_true, y_pred):
    y_pred = K.round(y_pred)
    tp = K.sum(K.cast(y_true * y_pred, 'float'), axis=0)
    fp = K.sum(K.cast((1 - y_true) * y_pred, 'float'), axis=0)
    fn = K.sum(K.cast(y_true * (1 - y_pred), 'float'), axis=0)

    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    
    f1 = 2 * precision * recall / (precision + recall + K.epsilon())
    return K.mean(f1)

def build_bilstm_model(num_vocabs, num_frames=19, num_landmarks=42):

    input_layer = Input(shape=(num_frames, num_landmarks*3))  
    
    # Pre-processing layers
    x = TimeDistributed(Dense(256, activation='relu', kernel_regularizer=l2(0.001)))(input_layer)
    x = LayerNormalization()(x)
    x = Dropout(0.3)(x)
    
    # First BiLSTM layer
    x = Bidirectional(
        LSTM(256, return_sequences=True, kernel_regularizer=l2(0.001)),
        merge_mode='concat'
    )(x)
    x = LayerNormalization()(x)
    x = Dropout(0.4)(x)
    
    # Second BiLSTM layer with attention
    lstm_out = Bidirectional(
        LSTM(256, return_sequences=True, kernel_regularizer=l2(0.001)),
        merge_mode='concat'
    )(x)
    lstm_out = LayerNormalization()(lstm_out)
    
    # Attention mechanism
    attention = Attention()([lstm_out, lstm_out])
    x = Concatenate()([lstm_out, attention])
    x = Dropout(0.5)(x)
    
    # Third BiLSTM layer
    x = Bidirectional(
        LSTM(256, return_sequences=False, kernel_regularizer=l2(0.001)),
        merge_mode='concat'
    )(x)
    x = LayerNormalization()(x)
    
    # Dense layers
    x = Dense(512, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    
    # Output layer
    output_layer = Dense(num_vocabs, activation='softmax')(x)
    
    # Create model
    model = Model(inputs=input_layer, outputs=output_layer)
    
    # Compile model
    optimizer = Adam(learning_rate=0.001, clipnorm=1.0)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=[
            CategoricalAccuracy(name='accuracy'),
            Precision(name='precision'),
            Recall(name='recall'),
            TopKCategoricalAccuracy(k=5, name='top5_accuracy'),
            f1_score
        ]
    )
    
    return model

In [22]:
# model = build_bilstm_model(num_vocabs = len(label), num_frames = sequence_length, num_landmarks = 42)
# model.load_weights("../models/final_model.keras")

C:\Users\USER\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 70 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
# model.fit(X_train, y_train, epochs=2000, callbacks=[tb_callback])

In [ ]:
# model.summary()

## Dataset Preprocess

In [24]:
label_map = {label:num for num, label in enumerate(label)}

In [21]:
sequences, labels = [], []
for action in label:
    for sequence in os.listdir(os.path.join(DATA_PATH, action)):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, sequence, "{}.npy".format(frame_num+1)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [23]:
np.array(sequences).shape

(1560, 19, 126)

In [25]:
np.array(labels).shape

(1560,)

## Train Model

so we use 1000 epochs with 128 batch size, training will hit early stop if it reached 100% accuracy either in train or validation set. saved model based on best validation accuracy. For learning rate, ReduceLROnPlateau automatically reduces the learning rate by a factor of 0.2 if the val_loss does not improve for 15 consecutive epochs, with a minimum limit of 1e-5.

In [ ]:
def train_model(X_train, y_train, X_val, y_val, num_vocabs, epochs = 1000, batch_size = 128):

    # Build model
    model = build_bilstm_model(num_vocabs)

    class StopAt100Acc(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            if logs.get('val_accuracy') == 1.0 or logs.get('accuracy') == 1.0:
                print("\nReached 100% val accuracy. Stopping training.")
                self.model.stop_training = True
    
    # Callbacks
    callbacks = [
        EarlyStopping(monitor='val_accuracy', patience=50, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=15, min_lr=1e-5),
        tf.keras.callbacks.ModelCheckpoint(
            'best_model_handlandmarks.keras',
            save_best_only=True,
            monitor='val_accuracy',
            mode='max'
        ),
        StopAt100Acc()
    ]
    
    # Train model
    with tf.device('/device:GPU:0'):
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1
        )
    
    return model, history

In [27]:
X = np.array(sequences)
y = to_categorical(labels).astype(int)

In [29]:
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=42, test_size=0.2)

In [31]:
X_train.shape

(1248, 19, 126)

In [33]:
y_train.shape

(1248, 13)

In [35]:
X_val.shape

(312, 19, 126)

In [ ]:
model, result = train_model(X_train = X_train, y_train = y_train, X_val = X_val, y_val = y_val, num_vocabs = len(label_train))

## Testing

In [32]:
cap = cv2.VideoCapture(0)

flag = 0
sequence = []
sentence = []
predictions = []
threshold = 0.5


with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        
                
        # Read feed
        ret, frame = cap.read()

        frame = cv2.flip(frame, 1)

        image, results = mediapipe_detection(frame, holistic)
        
        draw_styled_landmarks(image, results)
        
        if not flag:
            cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
            cv2.putText(image, ' '.join(sentence), (3,30), 
            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.rectangle(image, (50, 50), (380, 100), (0, 255, 0), -1)
            cv2.putText(image, "Press 's' to start recording", (60, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.imshow('Testing', image)
            key = cv2.waitKey(1) & 0xFF
            if key == ord('s'):
                print("Detection Triggered")
                flag = 1
                sequence = []
                sentence = []
                predictions = []
            elif key == 27 or key == ord('q'):
                cap.release()
                cv2.destroyAllWindows()
                break
            else:
                continue

        cv2.putText(image, f"taking sequence number {len(sequence)}", 
                    (image.shape[1] - cv2.getTextSize(f"taking sequence number {len(sequence)}", 
                                                      cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0][0] - 10, image.shape[0] - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        
        keypoints = extract_landmarks(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        for seq in sequence:
            if np.all(seq == 0):
                sequence = []
        
        if len(sequence) == sequence_length:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(res)
            print(label[np.argmax(res)])
            sequence = []
            predictions.append(np.argmax(res))
            
            
            if res[np.argmax(res)] > threshold: 
                
                if len(sentence) > 0: 
                    if label[np.argmax(res)] != sentence[-1]:
                        sentence.append(label[np.argmax(res)])
                        # flag ^= 1
                    else:
                        sequence = []
                else:
                    sentence.append(label[np.argmax(res)])
                    # flag ^= 1

            if len(sentence) > 5: 
                sentence = sentence[-5:]
            
        cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3,30), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.rectangle(image, (0, image.shape[0] - 40), (240, image.shape[0]), (245, 117, 16), -1)
        
        cv2.putText(image, "press 'p' to pause", (10, image.shape[0] - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        
       


        
        # Show to screen
        cv2.imshow('Testing', image)

        key_press = cv2.waitKey(1) & 0xFF

        if key_press == ord('q') or cv2.getWindowProperty('Testing', cv2.WND_PROP_VISIBLE) < 1:
            break
        elif key_press == ord('p'):
            flag = 0
    cap.release()
    cv2.destroyAllWindows()

Detection Triggered
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
[5.3269678e-04 6.0374509e-07 9.9943119e-01 1.2945672e-07 2.8984892e-05
 3.3255101e-06 5.2277545e-07 5.9493385e-07 7.0124301e-08 1.3310467e-09
 4.6696201e-08 1.8457376e-06 5.1809629e-10]
kamu
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
[9.2041582e-06 2.4242374e-05 1.2298636e-05 7.6638549e-05 2.5338109e-05
 5.6484323e-06 1.4689027e-05 2.2529447e-04 1.5423881e-04 1.9539418e-04
 9.9644011e-01 9.6598128e-04 1.8507633e-03]
suka
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
[1.0432628e-05 1.8902640e-05 1.2034249e-05 2.5975960e-05 1.2221592e-05
 1.5810136e-04 1.5839247e-04 2.1073675e-04 3.9715738e-05 4.8095928e-04
 2.1611148e-02 3.4704912e-04 9.7691435e-01]
aku
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
[6.5338718e-05 1.2233293e-05 8.9754549e-06 5.6735116e-05 2.3848932e-06
 3.4212109e-05 9.9964869e-01 1.3160668e-05 1.8368377e-05 2.9769730e-05
 7.5042676e-06 4.4866470e-06 9.8089899e-05]
siapa
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
[5.7275451e-05 1.3016849e-